# AXI test 5: the same test on a hosted Gemini model (free tier)
Same 140 requests, same 30-request episodes, same three policies and the same reading rules as the 14B test, but the model is **Gemini 3.5 Flash Lite**, reached through Google's free API. The prediction and the reading rules were written before this run in `docs/API_TEST_GEMINI.md`. It is Google's small, fast tier, not a Pro model; Part B at the end asks the 20 deciding requests of **Gemini 3.8 Flash**.

**What is different from the 14B test**
* **No GPU.** Use the free CPU runtime (Runtime menu -> *Change runtime type* -> CPU). Nothing is downloaded.
* **A daily limit.** The free key allows about 500 calls a day. The notebook stops by itself at 480 and says so. Everything finished is saved on your Google Drive, and so is every reply. **Run it again the next day**: open this notebook, run **Cells 2 to 5 again**, then run the cell you were on again (a finished cell ends in seconds). The limit resets at midnight Pacific time: **12:30 pm in India until 1 November, 1:30 pm after.** The whole of Part A takes **two to three days**.
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in any file. Never paste it in the chat.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

### Cell 1: put your key into Colab Secrets (once; no code to run)
1. In Google AI Studio (`aistudio.google.com`), open **API Keys** in the left menu and copy your key.
2. Here in Colab, tap the **key icon** in the left bar (*Secrets*) -> **Add new secret**.
3. Name: `GEMINI_API_KEY` (exactly). Value: paste the key. Switch **Notebook access** on.
Do not paste the key into a cell or into the chat.

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-gemini-v3.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini")

### Cell 5: connect to the model and check it with 6 requests (about 1 minute). It prints SMOKE OK or STOP. If it says STOP, do not run the next cells: send me the printout

In [ ]:
import json, time
from google.colab import userdata
from axi.experiments.api_llm import QuotaStop, ServiceBusy, build_gemini_llm
from axi.experiments.scope_tasks import make_scope_tasks
from axi.experiments.scope_test import run_scope_regime, summarize_scope, print_scope, print_exact, print_verdict
from axi.experiments.long_horizon import run_long, print_long
from axi.experiments.bigger_model import verdict_big_single, verdict_big_long, mean_seconds
from axi.experiments.api_test import verdict_api_format, verdict_pivotal, smoke_report

try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
WANTED = "gemini-3.5-flash-lite"
CACHE = "axi_gemini_cache.jsonl"
print("reply file:", store.restore(CACHE))
llm, MODEL = build_gemini_llm(KEY, WANTED, store.path(CACHE), daily_limit=480, min_interval=4.5, on_write=lambda: store.persist(CACHE))
RESET = "The free quota resets at midnight Pacific time: 12:30 pm in India until 1 November, 1:30 pm after. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once (if this page is still open, you do not need the earlier cells). Do NOT run it again and again: Google counts failed tries against the free daily limit."
T0 = time.time()
def prog(i, n): print(f"  {i}/{n} done   {time.time() - T0:6.0f} s   calls today: {llm.calls_today()}" + ("" if store.persistent else "   (NOT saved to Drive)"), end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e)); print(BUSY if isinstance(e, ServiceBusy) else RESET); return None
res_s_rule = res_s_norule = res_l_rule = res_l_norule = None
smoke = go(lambda: run_scope_regime(llm, make_scope_tasks(6, 7), True, "/content/smoke_single.jsonl", meta=dict(model=MODEL, smoke=True)))
if smoke:
    for r in smoke["rows"]: print(f"{r['kind']:14s} {r['seconds']:5.1f} s  {r['text'][:100]!r}")
    print("totals so far:", llm.usage())
    ok, msg = smoke_report(smoke["rows"]); print("\n" + msg)

### Cell 6: PART 1, rule stated in the prompt. 140 requests, one at a time (about 15 to 30 minutes; the first 6 are already done)

In [ ]:
T0 = time.time()
tasks = make_scope_tasks(140, 7)
name = "axi_gemini_single_rule"
print("progress file:", store.restore(name + ".jsonl"))
res_s_rule = go(lambda: run_scope_regime(llm, tasks, True, store.path(name + ".jsonl"), meta=dict(model=MODEL, seed=7), progress=store.progress(name + ".jsonl", prog)))
if res_s_rule:
    print(); print_scope(res_s_rule["summary"], "PART 1, rule stated, " + MODEL)
    print("mean seconds per reply:", round(mean_seconds(res_s_rule["rows"]), 2), "  calls today:", llm.calls_today())
    files.download(save(res_s_rule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 7: PART 1, no rule in the prompt. 140 requests (about 15 to 30 minutes)

In [ ]:
assert res_s_rule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = "axi_gemini_single_norule"
print("progress file:", store.restore(name + ".jsonl"))
res_s_norule = go(lambda: run_scope_regime(llm, tasks, False, store.path(name + ".jsonl"), meta=dict(model=MODEL, seed=7), progress=store.progress(name + ".jsonl", prog)))
if res_s_norule:
    print(); print_scope(res_s_norule["summary"], "PART 1, no rule, " + MODEL)
    print("mean seconds per reply:", round(mean_seconds(res_s_norule["rows"]), 2), "  calls today:", llm.calls_today())
    print_verdict(verdict_big_single({True: res_s_rule["summary"], False: res_s_norule["summary"]}) + verdict_api_format({True: res_s_rule["rows"], False: res_s_norule["rows"]}), "PART 1, both regimes")
    files.download(save(res_s_norule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 8: PART 2, rule stated. 6 episodes x 30 requests, 3 policies (about 30 to 60 minutes; may stop for the day and continue tomorrow)

In [ ]:
assert res_s_norule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = "axi_gemini_long_rule"
print("progress file:", store.restore(name + ".jsonl"))
res_l_rule = go(lambda: run_long(llm, "state", 6, 30, 7, True, store.path(name + ".jsonl"), meta=dict(model=MODEL), policies=("blind", "repair_m6", "scope_m7"), progress=store.progress(name + ".jsonl", prog), mix="scope"))
if res_l_rule:
    print(); print_long(res_l_rule["summary"], "PART 2, rule stated, " + MODEL); print_exact(res_l_rule["summary"], "PART 2, rule stated, " + MODEL)
    files.download(save(res_l_rule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 9: PART 2, no rule. 4 episodes x 30 requests (about 20 to 40 minutes). Prints all the reading-rule lines at the end

In [ ]:
assert res_l_rule, "The cell above stopped for today. Run it again first."
T0 = time.time()
name = "axi_gemini_long_norule"
print("progress file:", store.restore(name + ".jsonl"))
res_l_norule = go(lambda: run_long(llm, "state", 4, 30, 7, False, store.path(name + ".jsonl"), meta=dict(model=MODEL), policies=("blind", "repair_m6", "scope_m7"), progress=store.progress(name + ".jsonl", prog), mix="scope"))
if res_l_norule:
    print(); print_long(res_l_norule["summary"], "PART 2, no rule, " + MODEL); print_exact(res_l_norule["summary"], "PART 2, no rule, " + MODEL)
    print_verdict(verdict_big_long(res_l_rule["summary"], res_l_norule["summary"]), "PART 2, both regimes")
    print_verdict(verdict_api_format({True: res_s_rule["rows"], False: res_s_norule["rows"]}, {True: res_l_rule["rows"], False: res_l_norule["rows"]}), "format, all four files")
    print("totals:", llm.usage())
    files.download(save(res_l_norule, store.path(name + ".json"))); store.persist(name + ".json")
    files.download(store.path(CACHE))

### Cell 10: PART B, day 1. Gemini 3.8 Flash, the 20 deciding requests, no rule (20 calls = the whole day's free quota; about 5 minutes)
Part A does not have to be finished first. This cell uses its own reply file and its own limit.
**Run this cell once. If it says Google is busy, wait at least 30 minutes before running it again; every failed try can use up one of the 20 free calls of the day.**

In [ ]:
CACHE_B = "axi_gemini_cache_B.jsonl"
print("reply file:", store.restore(CACHE_B))
llmB, MODEL_B = build_gemini_llm(KEY, "gemini-3.8-flash", store.path(CACHE_B), daily_limit=20, min_interval=13, on_write=lambda: store.persist(CACHE_B))
pivotal = [t for t in make_scope_tasks(140, 7) if t.kind == "delete_dep"]
print(len(pivotal), "delete_dep requests")
def progB(i, n): print(f"  {i}/{n} done   calls today: {llmB.calls_today()}", end="\r")
def goB(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e)); print(BUSY if isinstance(e, ServiceBusy) else RESET); return None
name = "axi_gemini_B_norule"
print("progress file:", store.restore(name + ".jsonl"))
res_b_norule = goB(lambda: run_scope_regime(llmB, pivotal, False, store.path(name + ".jsonl"), meta=dict(model=MODEL_B, seed=7, pivotal=True), progress=store.progress(name + ".jsonl", progB)))
res_b_rule = None
if res_b_norule:
    print(); print_scope(res_b_norule["summary"], "PART B, no rule, " + MODEL_B)
    print_verdict(verdict_pivotal({False: res_b_norule["summary"]}), "PART B, no rule")
    files.download(save(res_b_norule, store.path(name + ".json"))); store.persist(name + ".json")

### Cell 11: PART B, day 2. The same 20 requests with the rule stated (20 calls; run this on the NEXT day, after the quota reset; first run Cells 2 to 4 and Cell 10 again)

In [ ]:
assert res_b_norule, "Run Cell 10 first (it stops for today if the quota is used)."
name = "axi_gemini_B_rule"
print("progress file:", store.restore(name + ".jsonl"))
res_b_rule = goB(lambda: run_scope_regime(llmB, pivotal, True, store.path(name + ".jsonl"), meta=dict(model=MODEL_B, seed=7, pivotal=True), progress=store.progress(name + ".jsonl", progB)))
if res_b_rule:
    print(); print_scope(res_b_rule["summary"], "PART B, rule stated, " + MODEL_B)
    print_verdict(verdict_pivotal({False: res_b_norule["summary"], True: res_b_rule["summary"]}), "PART B, both regimes")
    print("totals:", llmB.usage())
    files.download(save(res_b_rule, store.path(name + ".json"))); store.persist(name + ".json")
    files.download(store.path(CACHE_B))

### Done
Files downloaded: `axi_gemini_single_rule.json`, `axi_gemini_single_norule.json`, `axi_gemini_long_rule.json`, `axi_gemini_long_norule.json`, `axi_gemini_cache.jsonl` (Part A) and `axi_gemini_B_norule.json`, `axi_gemini_B_rule.json`, `axi_gemini_cache_B.jsonl` (Part B). Upload them to the chat, with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.